# Overcooked Ortamında İnsan-Yapay Zeka İşbirliği
## Bayesian Belief-Update Agent (BeliefAgentV2)

**Öğrenci**: Furkan Bora Ekerel — 211101053  
**Ders**: YAP 441 — Yapay Zeka

Bu notebook, BeliefAgentV2'nin çalışma prensibini, belief update mekanizmasını ve farklı agent'larla karşılaştırma sonuçlarını içermektedir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import matplotlib.pyplot as plt

from overcooked_ai_py.mdp.overcooked_mdp import OvercookedGridworld
from overcooked_ai_py.mdp.overcooked_env import OvercookedEnv
from overcooked_ai_py.agents.agent import RandomAgent, AgentPair
from overcooked_ai_py.agents.benchmarking import AgentEvaluator
from overcooked_ai_py.visualization.state_visualizer import StateVisualizer

from belief_agent_v2 import (
    BeliefAgentV2, INTENTS, WEIGHT_MATRIX, TRANSITION_TABLES, NUM_INTENTS,
)
from turn_based_evaluate import turn_based_evaluate

print("Import başarılı!")

## 1. Mimari Genel Bakış

BeliefAgentV2 bir **Gizli Markov Modeli (HMM)** tabanlı karar sistemidir:

- **Gizli Durum** $X_n$: İnsan oyuncunun niyeti (8 intent: GET_ONION, GET_TOMATO, PUT_ONION_IN_POT, PUT_TOMATO_IN_POT, GET_DISH, GET_SOUP, SERVE_SOUP, UNKNOWN)
- **Gözlem** $E_n$: 18 boyutlu binary vektör (pozisyon değişimi, nesne alma/bırakma, hareket yönü)
- **Belief Update**: $B(X_n) \propto P(E_n | X_n) \cdot B'(X_n)$ (Bayes kuralı)
- **Transition**: $B'(X_{n+1}) = T^T \cdot B(X_n)$ (olaylarla güncellenen geçiş modeli)

## 2. Weight Matrix (Ağırlık Matrisi)

Her intent için 18 feature'ın etkisini belirleyen $8 \times 18$ ağırlık matrisi. Pozitif değerler bir feature'ın o intent ile uyumlu olduğunu, negatif değerler uyumsuzluğu gösterir.

In [ ]:
# Weight Matrix Heatmap
from belief_agent_v2 import FEATURES

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(WEIGHT_MATRIX, cmap="RdYlGn", aspect="auto", vmin=-3, vmax=3)

ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(FEATURES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(NUM_INTENTS))
ax.set_yticklabels(INTENTS, fontsize=9)
ax.set_title("Weight Matrix W (8 intent × 18 feature)", fontsize=12)
ax.set_xlabel("Features (E_n)")
ax.set_ylabel("Intents (X_n)")

# Değerleri hücrelere yaz
for i in range(NUM_INTENTS):
    for j in range(len(FEATURES)):
        val = WEIGHT_MATRIX[i, j]
        if val != 0:
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(val) >= 2 else "black")

plt.colorbar(im, ax=ax, label="Ağırlık")
plt.tight_layout()
plt.show()

## 3. Transition Tables (Geçiş Matrisleri)

6 farklı olay için $8 \times 8$ geçiş matrisleri. Her olay insanın intent'ini nasıl değiştireceğini modeller.

In [ ]:
# Transition Tables Visualization
event_names = list(TRANSITION_TABLES.keys())
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, event in enumerate(event_names):
    T = TRANSITION_TABLES[event]
    ax = axes[idx]
    im = ax.imshow(T, cmap="Blues", vmin=0, vmax=1, aspect="equal")
    ax.set_title(f"Z = {event}", fontsize=10)
    ax.set_xticks(range(NUM_INTENTS))
    ax.set_xticklabels(INTENTS, rotation=45, ha="right", fontsize=6)
    ax.set_yticks(range(NUM_INTENTS))
    ax.set_yticklabels(INTENTS, fontsize=6)
    
    # Değerleri hücrelere yaz (0 hariç)
    for i in range(NUM_INTENTS):
        for j in range(NUM_INTENTS):
            val = T[i, j]
            if val > 0.05:
                ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=5,
                        color="white" if val > 0.5 else "black")

plt.suptitle("Geçiş Matrisleri T(X_{n+1} | X_n, Z_n)", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Dispenser Erişim Analizi

Farklı layout'larda hangi oyuncunun hangi dispenser'lara erişebildiğini gösterir. Bu bilgi agent'ın "insanın yapamayacağı işi ben yapmalıyım" kararı için kullanılır.

In [ ]:
from overcooked_ai_py.planning.planners import MediumLevelActionManager

layouts = ["forced_coordination_tomato", "forced_coordination", "asymmetric_advantages"]

fig, axes = plt.subplots(1, len(layouts), figsize=(5 * len(layouts), 4))

for idx, layout_name in enumerate(layouts):
    mdp = OvercookedGridworld.from_layout_name(layout_name)
    env = OvercookedEnv.from_mdp(mdp, horizon=100, info_level=0)
    state = env.state

    mlam = MediumLevelActionManager.from_pickle_or_compute(
        mdp,
        {"start_orientations": False, "wait_allowed": False,
         "counter_goals": [], "counter_drop": [], "counter_pickup": [],
         "same_motion_goals": True},
        force_compute=False,
    )
    mp = mlam.joint_motion_planner.motion_planner

    dispenser_map = {
        "onion": mdp.get_onion_dispenser_locations(),
        "tomato": mdp.get_tomato_dispenser_locations(),
    }
    ingredients = [k for k, v in dispenser_map.items() if v]

    # Her oyuncu × ingredient erişim tablosu
    data = np.zeros((2, len(ingredients)))
    for pidx in range(2):
        start = state.players[pidx].pos_and_or
        for j, ing in enumerate(ingredients):
            cost = mp.min_cost_to_feature(start, dispenser_map[ing])
            data[pidx, j] = 1 if cost < float("inf") else 0

    ax = axes[idx]
    im = ax.imshow(data, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ingredients)))
    ax.set_xticklabels(ingredients, fontsize=10)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Player 0\n(Human)", "Player 1\n(AI)"], fontsize=9)
    ax.set_title(layout_name, fontsize=10)
    for i in range(2):
        for j in range(len(ingredients)):
            txt = "✓" if data[i, j] == 1 else "✗"
            ax.text(j, i, txt, ha="center", va="center", fontsize=16,
                    color="white" if data[i, j] == 0 else "black")

plt.suptitle("Dispenser Erişim Haritası (Oyuncu × Ingredient)", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Karşılaştırma Deneyleri

Üç farklı layout üzerinde **BeliefAgentV2** ile **RandomAgent** karşılaştırması. Turn-based modda çalıştırılır.

| Metrik | Açıklama |
|--------|----------|
| **Toplam Skor** | Teslim edilen çorbaların toplam ödülü |
| **Adım Sayısı** | Oyunun kaç adım sürdüğü |
| **Skor/Adım** | Verimlilik ölçüsü |

In [ ]:
## -- Deney: BeliefAgentV2 (AI) + RandomAgent (Human) --

test_layouts = ["forced_coordination_tomato", "forced_coordination", "asymmetric_advantages"]
HORIZON = 400
NUM_GAMES = 3

results = {}

for layout_name in test_layouts:
    print(f"\n{'='*50}")
    print(f"Layout: {layout_name}")
    print(f"{'='*50}")

    # ------ BeliefAgentV2 + RandomAgent ------
    belief_agent = BeliefAgentV2()
    random_agent = RandomAgent()

    traj_belief = turn_based_evaluate(
        agent0=random_agent,    # Player 0 = Human (Random)
        agent1=belief_agent,    # Player 1 = AI (Belief)
        layout_name=layout_name,
        horizon=HORIZON,
        num_games=NUM_GAMES,
        display=False,
    )

    # ------ RandomAgent + RandomAgent (Baseline) ------
    random0 = RandomAgent()
    random1 = RandomAgent()

    traj_random = turn_based_evaluate(
        agent0=random0,
        agent1=random1,
        layout_name=layout_name,
        horizon=HORIZON,
        num_games=NUM_GAMES,
        display=False,
    )

    results[layout_name] = {
        "belief_returns": traj_belief["ep_returns"],
        "belief_lengths": traj_belief["ep_lengths"],
        "random_returns": traj_random["ep_returns"],
        "random_lengths": traj_random["ep_lengths"],
        "traj_belief": traj_belief,
        "traj_random": traj_random,
    }

    # Özet
    b_avg = np.mean(traj_belief["ep_returns"])
    r_avg = np.mean(traj_random["ep_returns"])
    print(f"  BeliefAgent + Random → Ortalama Skor: {b_avg:.1f}")
    print(f"  Random + Random      → Ortalama Skor: {r_avg:.1f}")
    print(f"  Fark: {b_avg - r_avg:+.1f}")

print("\nDeneyler tamamlandı!")

### 5.1 Skor Karşılaştırma Grafiği

Her layout için BeliefAgentV2 ve RandomAgent (baseline) ortalama skorları yan yana gösterilir.

In [ ]:
# Skor karşılaştırma bar chart
layout_names = list(results.keys())
belief_means = [np.mean(results[l]["belief_returns"]) for l in layout_names]
random_means = [np.mean(results[l]["random_returns"]) for l in layout_names]
belief_stds = [np.std(results[l]["belief_returns"]) for l in layout_names]
random_stds = [np.std(results[l]["random_returns"]) for l in layout_names]

x = np.arange(len(layout_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, belief_means, width, yerr=belief_stds,
               label="BeliefAgentV2 + Random", color="#2196F3", capsize=5)
bars2 = ax.bar(x + width/2, random_means, width, yerr=random_stds,
               label="Random + Random", color="#FF9800", capsize=5)

ax.set_xlabel("Layout")
ax.set_ylabel("Ortalama Skor")
ax.set_title("BeliefAgentV2 vs RandomAgent — Skor Karşılaştırması")
ax.set_xticks(x)
ax.set_xticklabels(layout_names, rotation=15, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Değerleri bar üzerine yaz
for bar in bars1:
    h = bar.get_height()
    ax.annotate(f"{h:.0f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.0f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### 5.2 Detaylı Sonuç Tablosu

In [ ]:
# Detaylı sonuç tablosu
print(f"{'Layout':<35} {'Agent Pair':<25} {'Ort. Skor':>10} {'Std':>8} {'Ort. Adım':>10} {'Skor/Adım':>10}")
print("=" * 100)

for layout_name in results:
    r = results[layout_name]
    # Belief row
    b_score = np.mean(r["belief_returns"])
    b_std = np.std(r["belief_returns"])
    b_steps = np.mean(r["belief_lengths"])
    b_eff = b_score / max(b_steps, 1)
    print(f"{layout_name:<35} {'BeliefV2 + Random':<25} {b_score:>10.1f} {b_std:>8.1f} {b_steps:>10.0f} {b_eff:>10.3f}")
    # Random row
    r_score = np.mean(r["random_returns"])
    r_std = np.std(r["random_returns"])
    r_steps = np.mean(r["random_lengths"])
    r_eff = r_score / max(r_steps, 1)
    print(f"{'':<35} {'Random + Random':<25} {r_score:>10.1f} {r_std:>8.1f} {r_steps:>10.0f} {r_eff:>10.3f}")
    # Improvement
    improv = b_score - r_score
    print(f"{'':<35} {'→ İyileşme':<25} {improv:>10.1f}")
    print("-" * 100)

## 6. Belief Tracking — İnanç Vektörünün Zaman İçinde Değişimi

Bir oyun sırasında agent'ın insan niyetine dair inanç vektörü $B(X_n)$ nasıl değişiyor? Bu grafik, her adımda her intent'in posterior olasılığını gösterir.

In [ ]:
# Belief vektörünü adım adım izle (forced_coordination_tomato üzerinde)
from overcooked_ai_py.mdp.actions import Action

layout_name = "forced_coordination_tomato"
mdp = OvercookedGridworld.from_layout_name(layout_name)
env = OvercookedEnv.from_mdp(mdp, horizon=100, info_level=0)
env.reset()

belief_agent = BeliefAgentV2()
random_agent = RandomAgent()

random_agent.reset()
belief_agent.reset()
random_agent.set_agent_index(0)
belief_agent.set_agent_index(1)
belief_agent.set_mdp(mdp, initial_state=env.state)

# Posterior'ları kaydet
belief_history = []
TRACK_STEPS = 50

for t in range(TRACK_STEPS):
    state = env.state
    # Human (random) hareket
    a0, _ = random_agent.action(state)
    env.step((a0, Action.STAY))

    # AI hareket — bu sırada belief güncellenir
    state = env.state
    a1, _ = belief_agent.action(state)
    env.step((Action.STAY, a1))

    # Posterior'ı kaydet
    belief_history.append(belief_agent._posterior.copy())

belief_history = np.array(belief_history)  # (TRACK_STEPS, 8)

# Çiz
fig, ax = plt.subplots(figsize=(14, 5))
for i, intent in enumerate(INTENTS):
    ax.plot(belief_history[:, i], label=intent, linewidth=1.5)

ax.set_xlabel("Adım (t)")
ax.set_ylabel("P(X_n = intent)")
ax.set_title(f"Belief Tracking — {layout_name} ({TRACK_STEPS} adım)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Sonuç ve Değerlendirme

### Mimari Özet
BeliefAgentV2, Overcooked ortamında insanın niyetini **Bayesian belief update** ile tahmin eden bir yapay zeka agentıdır:

1. **Gözlem Katmanı (Katman 0)**: İnsanın hamlesinden 18 binary feature çıkarır
2. **Belief Update**: $B(X_n) \propto P(E_n|X_n) \cdot B'(X_n)$ — Bayes kuralıyla posterior hesaplar
3. **Transition Model**: 6 farklı olay için ayrı geçiş matrisleri — $B'(X_{n+1}) = T^T \cdot B(X_n)$
4. **Karar Katmanı (Katman 1+2)**: Fiziksel kurallar + belief-tabanlı stratejik kararlar

### Temel Özellikler
- **Dispenser Erişim Kontrolü**: İnsanın erişemediği dispenser'ları otomatik olarak AI devralır
- **Midpoint Counter Drop**: İki oyuncunun ortasındaki counter'a item bırakma
- **None-safe Navigation**: Erişilemeyen hedefler için fallback zinciri
- **Unstuck Mekanizması**: Tekrarlayan hamlelerden çıkış

### Kısıtlamalar
- RandomAgent ile test edildiğinde gerçek insan davranışı simüle edilmemektedir
- Weight matrix ve transition table'lar el ile tasarlanmıştır (öğrenilmemiştir)
- Turn-based mod, eşzamanlı hareketten farklı dinamikler oluşturabilir

---
**Proje**: YAP 441 — Yapay Zeka  
**Öğrenci**: Furkan Bora Ekerel — 211101053